# Contrasting signal smoothers

This tutorial compares `nelpy`'s established Gaussian smoother with the Chronux-style local linear smoother added as `nel.utils.locsmooth`. It also shows `nel.utils.savgol_filter`, a support-aware Savitzky–Golay filter that fits a local polynomial around every sample and can return smooth derivatives.

Most examples below use continuously sampled analog data. The last two sections show why these smoothers are a poor fit for binned spike counts, and how to estimate firing rates from spike times instead.

> **Warning:** `locsmooth` and `savgol_filter` can return *negative* firing rates when applied to binned spike trains. Estimate firing rates from the spike times with `nel.utils.kernel_rate` instead (see the last section).

## Choose a smoother by the scientific question

| Method | Main parameter | What it estimates | Good use |
| --- | --- | --- | --- |
| `nel.utils.gaussian_filter` | `sigma` | Kernel-weighted average | Smooth, low-pass denoising or spike-density-like signals |
| `nel.utils.locsmooth` | `window`, `step` | Overlapping local linear fits | Preserve locally linear trends, especially near an epoch boundary |
| `nel.utils.savgol_filter` | `window`, `polyorder`, `deriv` | Local polynomial fit centered on every sample | Preserve broad peaks and compute smooth derivatives |
| `nel.utils.kernel_rate` | `ds`, `sigma` (from `nel.utils.kernel_bandwidth`) | Firing rate from exact spike times | Spike trains: never negative, bandwidth chosen from the spikes |

`locsmooth` is inspired by Chronux's running-line routine. It is not the same as general LOESS/LOWESS: its fitting windows have a fixed duration, each window is an ordinary least-squares line fit, and a tricube taper blends adjacent fitted lines.

`savgol_filter` is a different estimator even at `polyorder=1`: it fits a window centered on every sample and keeps only the center value (at degree 1 that is a plain moving average), whereas `locsmooth` blends whole fitted lines from windows spaced `step` apart.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import nelpy as nel

## A trend, a transient, and measurement noise

The underlying signal has a slow linear drift, an oscillation, and a short transient. The drift makes the difference between kernel averaging and local linear fitting visible at the boundaries.

In [ ]:
fs = 200
time = np.arange(0, 6, 1 / fs)
rng = np.random.default_rng(4)

trend = 0.15 * time
oscillation = 0.35 * np.sin(2 * np.pi * 2.5 * time)
transient = np.exp(-0.5 * ((time - 3.4) / 0.08) ** 2)
observed = trend + oscillation + transient + rng.normal(scale=0.25, size=time.size)

asa = nel.AnalogSignalArray(observed, timestamps=time, fs=fs)
gaussian = nel.utils.gaussian_filter(asa, sigma=0.05)
running_line = nel.utils.locsmooth(asa, window=0.25)
savgol = nel.utils.savgol_filter(asa, window=0.25, polyorder=2)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(time, observed, color="0.75", lw=0.8, label="observed")
ax.plot(time, gaussian.data[0], lw=2, label=r"Gaussian ($\sigma$ = 50 ms)")
ax.plot(time, running_line.data[0], lw=2, label="running line (250 ms window)")
ax.plot(time, savgol.data[0], lw=2, label="Savitzky–Golay (250 ms window, degree 2)")
ax.set(xlabel="time (s)", ylabel="signal")
ax.legend(frameon=False, ncol=2)
plt.show()

The Gaussian result is a weighted average, so it tends to flatten a local slope near a boundary. `locsmooth` explicitly fits a local line and therefore retains a linear signal exactly. Savitzky–Golay uses a higher-order local polynomial, which can better retain peak shape but may introduce oscillatory-looking behavior when its window or polynomial degree is too large.

`window` has the same time unit as the signal. `step` defaults to half the window and controls how densely local lines are fitted; it does not downsample the returned signal.

## Support boundaries are never crossed

Nelpy signals may have disjoint support intervals. `locsmooth` and `savgol_filter` fit each interval independently, so a large level shift across an acquisition gap cannot influence either edge.

In [ ]:
left_time = np.arange(0, 1, 1 / fs)
right_time = np.arange(2, 3, 1 / fs)
segmented_time = np.r_[left_time, right_time]
segmented_data = np.r_[
    0.4 * left_time + rng.normal(scale=0.08, size=left_time.size),
    3 + 0.4 * (right_time - 2) + rng.normal(scale=0.08, size=right_time.size),
]
segmented = nel.AnalogSignalArray(
    segmented_data,
    timestamps=segmented_time,
    fs=fs,
    support=nel.EpochArray([[0, 1], [2, 3]]),
)
segmented_smooth = nel.utils.locsmooth(segmented, window=0.25)
segmented_savgol = nel.utils.savgol_filter(segmented, window=0.25)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(segmented.abscissa_vals, segmented.data[0], ".", color="0.7", label="observed")
ax.plot(segmented_smooth.abscissa_vals, segmented_smooth.data[0], lw=2, label="locsmooth")
ax.plot(segmented_savgol.abscissa_vals, segmented_savgol.data[0], lw=2, label="savgol_filter")
ax.axvspan(1, 2, color="0.95", zorder=-1, label="outside support")
ax.set(xlabel="time (s)", ylabel="signal")
ax.legend(frameon=False)
plt.show()

## Practical parameter selection

Start with `window` near the shortest timescale that should be considered signal rather than noise. Keep `step=window / 2` unless profiling demonstrates a reason to change it. Compare the result against unsmoothed data and report both parameters in analysis methods. For `savgol_filter`, keep `polyorder` low (2–3): at a fixed `window`, a higher degree follows the noise more closely.

`locsmooth` and `savgol_filter` assume samples on a uniform grid. If your samples are irregularly spaced (for example, a behavioral variable measured only at trial times), use a LOESS implementation instead: it fits each local regression on the actual sample times, and its window can be a fixed number of neighbors rather than a fixed duration. LOESS is not a spike-rate estimator, though. It is still a local regression on counts, and the robustness iterations that most implementations run by default treat isolated spikes as outliers and shrink the rate toward zero. For spike trains, use `nel.utils.kernel_rate`, shown at the end of this tutorial.

## Smooth derivatives

Savitzky–Golay's distinctive feature is `deriv`: it differentiates the fitted polynomial rather than the noisy samples. Derivatives are returned per `base_unit` (here, per second), so `deriv=1` on a position signal in cm gives a velocity in cm/s. A finite difference such as `asa.ddt()` amplifies sample-to-sample noise instead; its trace is clipped below because it swings by more than ±100 per second.

In [ ]:
true_slope = (
    0.15
    + 0.35 * 2 * np.pi * 2.5 * np.cos(2 * np.pi * 2.5 * time)
    - (time - 3.4) / 0.08**2 * transient
)
finite_difference = asa.ddt()
savgol_slope = nel.utils.savgol_filter(asa, window=0.25, polyorder=3, deriv=1)

fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(time, finite_difference.data[0], color="0.8", lw=0.8, label="finite difference (asa.ddt())")
ax.plot(time, savgol_slope.data[0], lw=2, color="C2", label="Savitzky–Golay, deriv=1 (250 ms window, degree 3)")
ax.plot(time, true_slope, "k--", lw=1.5, label="true derivative")
ax.set(xlabel="time (s)", ylabel="signal / s", ylim=(-25, 25))
ax.legend(frameon=False, ncol=3, loc="upper left")
plt.show()

## Regularly sampled analog data vs. sparse point-process data

All three smoothers above assume there is an underlying *smooth* function to recover: a kernel average, a local line, or a local polynomial all make sense when the observed samples are noisy measurements of something continuous, like an LFP channel. A spike train is not that. After binning, most bins are exactly 0 and the rest hold a small integer count — there's no smooth process to fit a local polynomial *to*, only a realization of a point process.

The cell below bins a handful of spikes at the same sampling rate as the continuous example and runs all three filters on both. Neither `locsmooth` nor `savgol_filter` accepts a `BinnedSpikeTrainArray` (only a `RegularlySampledAnalogSignalArray`), so the counts are wrapped in an `AnalogSignalArray` first — itself a sign that both were designed with continuous signals in mind, not event counts.

> **Warning:** `locsmooth` and `savgol_filter` can return *negative* firing rates when applied to binned spike trains. Estimate firing rates from the spike times with `nel.utils.kernel_rate` instead (see the last section).

In [ ]:
spike_times = np.sort(rng.uniform(0, 2, size=12))
st = nel.SpikeTrainArray(spike_times, support=nel.EpochArray([0, 2]), fs=fs)
bst = st.bin(ds=1 / fs)
counts = bst.data[0].astype(float)

window = 0.125  # 25 samples at fs = 200
continuous_sg = nel.utils.savgol_filter(asa, window=window, polyorder=3).data[0]
continuous_ls = nel.utils.locsmooth(asa, window=window).data[0]
continuous_gf = nel.utils.gaussian_filter(asa, sigma=0.05).data[0]

counts_asa = nel.AnalogSignalArray(counts, fs=fs, support=bst.support)
counts_sg = nel.utils.savgol_filter(counts_asa, window=window, polyorder=3).data[0]
counts_ls = nel.utils.locsmooth(counts_asa, window=window).data[0]
counts_gf = nel.utils.gaussian_filter(bst, sigma=0.05).data[0]

for name, trace in [("Savitzky-Golay", counts_sg), ("locsmooth", counts_ls), ("Gaussian", counts_gf)]:
    print(f"{name:>16} on spike counts: min={trace.min():+.3f}, negative samples={int((trace < 0).sum())}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(time, observed, color="0.75", lw=0.8, label="observed")
axes[0].plot(time, continuous_gf, lw=2, label="Gaussian")
axes[0].plot(time, continuous_ls, lw=2, label="locsmooth")
axes[0].plot(time, continuous_sg, lw=2, label="Savitzky\u2013Golay")
axes[0].set(ylabel="signal", title="Regularly sampled continuous data (e.g. LFP)")
axes[0].legend(frameon=False, ncol=4)

axes[1].vlines(spike_times, 0, 1, color="0.4", lw=1, label="spike times")
axes[1].plot(bst.bin_centers, counts_gf, lw=2, label="Gaussian")
axes[1].plot(bst.bin_centers, counts_ls, lw=2, label="locsmooth")
axes[1].plot(bst.bin_centers, counts_sg, lw=2, label="Savitzky\u2013Golay")
axes[1].axhline(0, color="0.85", lw=1, zorder=-1)
axes[1].set(xlabel="time (s)", ylabel="count / density",
            title="Sparse point-process data (binned spike train)")
axes[1].legend(frameon=False, ncol=4)
plt.tight_layout()
plt.show()

On the continuous signal (top), all three curves track each other closely, as they did in the very first comparison. On the binned spike counts (bottom), both `savgol_filter` and `locsmooth` swing below zero next to isolated spikes (negative samples printed above) — a local polynomial fit to a single 1 among a run of 0s has to come back down on both sides, and it overshoots past zero to do it. A negative spike *count* is meaningless, so neither filter's output is a valid density estimate here, even though both are well-behaved on the LFP-like signal above. `gaussian_filter`'s kernel average stays non-negative by construction. The next section takes that idea to its conclusion: put the kernel on the spike times themselves, and choose its width from the spikes.

## Estimating firing rates from spike times

A spike train is a set of event times generated by an underlying rate $\lambda(t) \ge 0$. `nel.utils.kernel_rate` estimates that rate by placing a Gaussian density on every spike time, $\hat\lambda(t) = \sum_i K_\sigma(t - t_i)$. This respects the point-process nature of the data:

- the estimate is a sum of non-negative bumps, so it **cannot go negative**;
- it is in spikes per second; each truncated kernel has unit area before support clipping or boundary correction;
- it uses the exact spike times; `ds` only sets where the estimate is evaluated, not where spikes are rounded to;
- spikes never contribute across a support gap. At each evaluation point, boundary correction divides by the kernel mass inside the interval, correcting bias for a locally constant rate without guaranteeing preservation of total spike count.

The kernel width comes from the spikes alone: `nel.utils.kernel_bandwidth` picks the $\sigma$ that minimizes Shimazaki & Shinomoto's (2010) estimate of the integrated squared error between $\hat\lambda$ and the unknown $\lambda$, assuming Poisson spiking.

To keep the comparison honest, the cell below simulates spikes from a known rate. `locsmooth`, `savgol_filter` and `gaussian_filter` each get their best possible width, chosen by comparing against the true rate, which you never have with real data. `kernel_rate` only sees the spikes.

In [ ]:
def true_rate(t):
    return (
        2
        + 25 * np.exp(-0.5 * ((t - 5) / 0.6) ** 2)
        + 15 * np.exp(-0.5 * ((t - 12) / 0.3) ** 2)
        + 8 * (t > 16)
    )


# Inhomogeneous Poisson spikes by thinning a 35 Hz homogeneous process.
duration, peak_rate = 20, 35
proposals = np.sort(rng.uniform(0, duration, rng.poisson(peak_rate * duration)))
keep = rng.uniform(0, peak_rate, proposals.size) < true_rate(proposals)
sim_spike_times = proposals[keep]
sim_st = nel.SpikeTrainArray(
    sim_spike_times, support=nel.EpochArray([0, duration]), fs=fs
)

ds = 0.01
sim_bst = sim_st.bin(ds=ds)
t_sim = sim_bst.bin_centers
truth = true_rate(t_sim)
counts_hz = nel.AnalogSignalArray(
    sim_bst.data[0] / ds, abscissa_vals=t_sim, fs=1 / ds, support=sim_bst.support
)


def rmse(estimate):
    return np.sqrt(np.mean((estimate - truth) ** 2))


widths = np.geomspace(0.05, 4, 40)
smoothers = {
    "gaussian_filter": lambda w: nel.utils.gaussian_filter(counts_hz, sigma=w),
    "locsmooth": lambda w: nel.utils.locsmooth(obj=counts_hz, window=w),
    "savgol_filter": lambda w: nel.utils.savgol_filter(counts_hz, window=w),
}
estimates, settings = {}, {}
for name, smooth in smoothers.items():
    # Oracle: pick the width that best matches the true rate.
    errors = [rmse(smooth(w).data[0]) for w in widths]
    best = widths[np.argmin(errors)]
    estimates[name] = smooth(best).data[0]
    settings[name] = f"width {best:.2f} s, tuned against the true rate"

sigma = nel.utils.kernel_bandwidth(sim_st)[0]
estimates["kernel_rate"] = nel.utils.kernel_rate(sim_st, ds=ds, sigma=sigma).data[0]
settings["kernel_rate"] = f"sigma {sigma:.2f} s, from kernel_bandwidth (spikes only)"

print(f"{sim_st.n_events[0]} spikes in {duration} s\n")
for name, estimate in estimates.items():
    print(
        f"{name:>15}: RMSE {rmse(estimate):4.2f} Hz | min {estimate.min():+6.2f} Hz "
        f"| negative bins {np.mean(estimate < 0):5.1%} | {settings[name]}"
    )

In [ ]:
sigmas = np.geomspace(0.05, 4, 40)
kernel_errors = [
    rmse(nel.utils.kernel_rate(sim_st, ds=ds, sigma=s).data[0]) for s in sigmas
]
colors = {
    "gaussian_filter": "C0",
    "locsmooth": "C1",
    "savgol_filter": "C2",
    "kernel_rate": "C3",
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [2, 1]})
ax = axes[0]
ax.plot(t_sim, truth, "k--", lw=1.5, label="true rate")
for name, estimate in estimates.items():
    ax.plot(
        t_sim,
        estimate,
        color=colors[name],
        lw=2.5 if name == "kernel_rate" else 1.2,
        label=name,
    )
ax.plot(
    sim_spike_times,
    np.full(sim_spike_times.size, -5),
    "|",
    color="0.3",
    ms=8,
    label="spikes",
)
ax.axhline(0, color="0.8", lw=1, zorder=-1)
ax.set(
    xlabel="time (s)",
    ylabel="rate (Hz)",
    title="Firing-rate estimates vs. the true rate",
)
ax.legend(frameon=False, ncol=3, loc="upper right")

ax = axes[1]
ax.plot(sigmas, kernel_errors, color="C3")
ax.axvline(sigma, color="C3", ls=":", label=f"kernel_bandwidth choice ({sigma:.2f} s)")
ax.set(
    xscale="log",
    xlabel="kernel sigma (s)",
    ylabel="RMSE vs. truth (Hz)",
    title="kernel_rate error across bandwidths",
)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

What the comparison shows:

- **Accuracy is similar.** Even with their widths tuned against the true rate, which is impossible with real data, the three smoothers reach about the same error as `kernel_rate` (RMSE printed above). The bottom panel shows that `kernel_bandwidth`, using only the spikes, chose a $\sigma$ close to the bottom of an error curve that only an oracle could draw.
- **Validity is not.** Even at their best widths, `locsmooth` and `savgol_filter` return negative firing rates in some bins (`min` and `negative bins` above). A kernel rate is a sum of non-negative bumps and cannot.
- **The width comes from the data.** With real recordings there is no true rate to tune `window` against. `kernel_bandwidth` gives each unit its own width from its own spikes, and `kernel_rate` accepts one `sigma` per unit.

`gaussian_filter` on binned counts is the same kernel idea and behaves much like `kernel_rate` at the same $\sigma$. It rounds spikes to bins first (harmless when `ds` is much smaller than $\sigma$) and still needs its $\sigma$ from somewhere; `kernel_bandwidth` is what supplies it.